In [ ]:
import json
import math
import os
import numpy as np
from scipy.sparse import load_npz
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
from mistralai import Mistral


In [ ]:
from scipy.stats import gamma, norm
from scipy.special import digamma, polygamma

In [ ]:
from scipy.stats import chi2

## BM25 distribution using query

In [ ]:
matrices_folder = "../NMF_code/articles_raw_proc_data/matrices"
data_path = "../articles_raw_proc_data/processed_data/guardian_train.json"
field = "text_all_lemma"

#first query: general banking / finance words
q1 = [
    "bank", "banker", "banking", "fine", "financial", "executive",
    "regulator", "chief", "institution", "scandal", "deposit"
]

q2_extras = [
    "settle", "settlement", "justice", "prosecution", "lawsuit",
    "sanction", "rig", "rigging", "compliance"
]

q2 = []
for word in q1:
    q2.append(word)
for word in q2_extras:   # ← loop over the extras
    if word not in q2:   # ← check against q2
        q2.append(word)

print("q2 terms: " + str(q2))
print("number of q2 terms: " + str(len(q2)))

#storing the queries in a plain dict so we can loop over them later
query_dict = {}
query_dict["High-recall, high-precision query"] = q1
query_dict["Enriched query with low-recall, high-precision set"] = q2

#colors for the two histograms — one per query
query_names_in_order = []
for name in query_dict:
    query_names_in_order.append(name)
colors_in_order = ["#2166ac", "#1a9641"]

#BM25 PARAMETERS

k1 = 1.5
b  = 0.75


#LOADING THE DATA - matrix, vocabulary, and the articles json

#build the file paths by hand using string concatenation
dtm_file_path   = matrices_folder + "/dtm_" + field + ".npz"
vocab_file_path = matrices_folder + "/vocab_" + field + ".npy"
dtm = load_npz(dtm_file_path)

#allow_pickle=True is needed for object arrays 
vocab = np.load(vocab_file_path, allow_pickle=True)
#loading the articles json 
print("loading articles from: " + data_path)
f = open(data_path, "r", encoding="utf-8")
docs = json.load(f)
f.close()

#BUILDING A TERM-TO-INDEX LOOKUP DICTIONARY

term2idx = {}
idx = 0
for term in vocab:
    term2idx[term] = idx
    idx = idx + 1

#total number of documents (rows in the matrix)
num_docs = dtm.shape[0]

#document lengths = number of tokens in each document
#dtm.sum(axis=1) gives a matrix column so we flatten it to a 1d array
doc_lengths = np.array(dtm.sum(axis=1)).ravel()
print("computed document lengths, shape: " + str(doc_lengths.shape))

#average document length across the whole corpus
total_token_count = 0.0
for dl in doc_lengths:
    total_token_count = total_token_count + dl
avg_doc_length = total_token_count / num_docs
print("average document length: " + str(round(avg_doc_length, 2)))

#document frequency for each term = number of docs the term appears in
#(dtm > 0) gives a boolean matrix, summing over rows gives count per column
df_counts = np.array((dtm > 0).sum(axis=0)).ravel()
print("computed df_counts, shape: " + str(df_counts.shape))

#convert dtm to CSR format for fast column slicing later
#CSR = compressed sparse row
dtm_csr = dtm.tocsr()
print("converted dtm to csr format")

#BM25 SCORING FUNCTION
#given a list of query terms, returns a score for every document
#higher score = document is more relevant to the query

def compute_bm25_scores(query_terms):
    scores = np.zeros(num_docs)

    for term_index in range(len(query_terms)):
        term = query_terms[term_index]
        if term not in term2idx:
            print("  skipping term not in vocab: " + term)
            continue
        col = term2idx[term]
        df_t = df_counts[col]

        #the +0.5 smoothing is standard BM25
        idf_numerator   = num_docs - df_t + 0.5
        idf_denominator = df_t + 0.5
        idf = math.log((idf_numerator / idf_denominator) + 1)
        tf_raw = np.array(dtm_csr[:, col].todense()).ravel()
        length_norm  = 1 - b + b * (doc_lengths / avg_doc_length)
        tf_numerator = tf_raw * (k1 + 1)
        tf_denominator = tf_raw + k1 * length_norm
        tf_norm = tf_numerator / tf_denominator
        scores = scores + idf * tf_norm

    return scores

#RUNNING BM25 FOR EACH QUERY

#storing results in a plain dict keyed by query name
all_scores = {}

for i in range(len(query_names_in_order)):
    name = query_names_in_order[i]
    terms = query_dict[name]
    print("scoring query: " + name)
    print("  number of terms: " + str(len(terms)))
    scores_for_this_query = compute_bm25_scores(terms)
    all_scores[name] = scores_for_this_query
    num_nonzero = 0
    for s in scores_for_this_query:
        if s > 0:
            num_nonzero = num_nonzero + 1
    print("  documents with score > 0: " + str(num_nonzero))

#BUILDING THE PLOTLY FIGURE 

#make_subplots creates a figure with 2 rows and 1 column
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=query_names_in_order,  #titles come from the query names
    vertical_spacing=0.12
)

#loop over each query by index so i can track row number and color
for i in range(len(query_names_in_order)):
    name  = query_names_in_order[i]
    color = colors_in_order[i]

    #plotly uses 1-based row numbers
    row_number = i + 1

    #get the scores array for this query
    s = all_scores[name]

    #collect only the non-zero scores — documents that matched at least one term
    nonzero_scores = []
    for score_val in s:
        if score_val > 0:
            nonzero_scores.append(score_val)

    #convert to numpy array so we can use np.histogram and .mean() etc. later
    nonzero_scores = np.array(nonzero_scores)

    print("plotting query " + str(row_number) + ": " + name)
    print("  non-zero scores: " + str(len(nonzero_scores)))

    #add the histogram trace for this query
    fig.add_trace(
        go.Histogram(
            x=nonzero_scores,
            nbinsx=80,
            marker_color=color,
            opacity=0.5,
            showlegend=False
        ),
        row=row_number,
        col=1
    )

    #figure out the correct axis reference strings for the annotation
    #plotly uses "x" and "y" for the first subplot and "x2", "y2" for the second etc.
    if row_number == 1:
        xref_str = "x"
        yref_str = "y"
    else:
        xref_str = "x" + str(row_number)
        yref_str = "y" + str(row_number)

    #compute the histogram bin heights so we know where to put the annotation
    #np.histogram returns (counts, bin_edges) — we only need the counts
    hist_result  = np.histogram(nonzero_scores, bins=80)
    hist_counts  = hist_result[0]  #first element is the counts array
    #find the tallest bar height so we can anchor the annotation near the top
    #doing this manually instead of using .max()
    y_max = hist_counts[0]
    for count_val in hist_counts:
        if count_val > y_max:
            y_max = count_val

    #compute summary stats for the annotation text
    total_score = 0.0
    for sv in nonzero_scores:
        total_score = total_score + sv
    mean_score = total_score / len(nonzero_scores)

    max_score = nonzero_scores[0]
    for sv in nonzero_scores:
        if sv > max_score:
            max_score = sv

    n_str    = str(len(nonzero_scores))  #will add comma formatting below
    n_formatted    = f"{len(nonzero_scores):,}"
    mean_formatted = str(round(mean_score, 1))
    max_formatted  = str(round(max_score, 1))
    terms_count    = str(len(query_dict[name]))

    annotation_text = ("n=" + n_formatted + "  " +
                       "mean=" + mean_formatted + "  " +
                       "max=" + max_formatted + "  " +
                       "|terms|=" + terms_count)

    x_position = max_score * 0.98
    y_position = y_max * 0.98

    fig.add_annotation(
        text=annotation_text,
        xref=xref_str,
        yref=yref_str,
        x=x_position,
        y=y_position,
        showarrow=False,
        font=dict(size=10),
        xanchor="right",
        yanchor="top"
    )

#update overall figure layout
fig.update_layout(
    height=900,
    width=900,
    title_text="BM25 score distribution — gradual query expansion",
)


save_path = "bm25_expansion.png"
fig.write_image(save_path, width=900, height=900, scale=2)
print("saved figure to: " + save_path)

fig.show()

print("")
print("=== SUMMARY ===")

for i in range(len(query_names_in_order)):
    name = query_names_in_order[i]
    s    = all_scores[name]
    #collect nonzero scores again for summary stats
    nz_list = []
    for score_val in s:
        if score_val > 0:
            nz_list.append(score_val)

    #compute mean manually
    total_for_mean = 0.0
    for sv in nz_list:
        total_for_mean = total_for_mean + sv
    mean_val = total_for_mean / len(nz_list)
    #compute max manually
    max_val = nz_list[0]
    for sv in nz_list:
        if sv > max_val:
            max_val = sv

    num_terms    = len(query_dict[name])
    num_nonzero  = len(nz_list)

    print(name)
    print("  terms=" + str(num_terms) +
          "  non-zero=" + f"{num_nonzero:,}" +
          "  mean=" + str(round(mean_val, 2)) +
          "  max=" + str(round(max_val, 2)))

Appending BM25 scores to guardian_launder_processed.json

In [ ]:

proc_path = "../articles_raw_proc_data/processed_data/guardian_train.json"
scores = all_scores["Enriched query with low-recall, high-precision set"]

#BUILD A LOOKUP DICTIONARY: doc index -> bm25 score

score_lookup = {}
i = 0
for doc in docs:
    doc_index = doc["index"]
    raw_score = scores[i]
    python_float = float(raw_score)
    rounded_score = round(python_float, 4)
    score_lookup[doc_index] = rounded_score
    i = i + 1
    if i % 10000 == 0:
        print("  processed " + str(i) + " docs into lookup so far...")

print("lookup dictionary built, total entries: " + str(len(score_lookup)))

#LOAD THE PROCESSED DOCS FROM DISK

print("loading processed docs from: " + proc_path)
f = open(proc_path, "r", encoding="utf-8")
proc_docs = json.load(f)
f.close()

print("loaded " + str(len(proc_docs)) + " processed documents")

#ADD BM25 SCORE TO EACH DOCUMENT

print("adding bm25 scores to each document...")

for doc in proc_docs:
    this_doc_index = doc["index"]
    this_score = score_lookup.get(this_doc_index, None)
    #attach the score as a new field on the document dict
    doc["bm25_score"] = this_score

print("scores attached to all documents")

#WRITE THE ENRICHED DOCS BACK TO THE SAME FILE

print("writing enriched documents back to: " + proc_path)
f = open(proc_path, "w", encoding="utf-8")
json.dump(proc_docs, f, ensure_ascii=False, indent=2)
f.close()

print("file saved successfully")

#SANITY CHECK — count how many documents actually got a non-zero score
total_docs = len(proc_docs)
#count docs with a non-zero bm25 score manually
nonzero_count = 0
for d in proc_docs:
    #need to check for None too, not just zero
    if d["bm25_score"] is not None and d["bm25_score"] > 0:
        nonzero_count = nonzero_count + 1

print("Done — " + f"{total_docs:,}" + " documents enriched")
print("Non-zero scores: " + f"{nonzero_count:,}")

Inspecting tail in enriched query BM25 plot

In [ ]:
threshold_check = 20.0
boundary = 20.0
boundary_window = 20

#COLLECT ALL DOCS THAT SCORE AT OR ABOVE THE THRESHOLD

above = []
for d in proc_docs:
    if d["bm25_score"] >= threshold_check:
        #store a tuple of (index, id, score) so we have everything we need later
        #i'm using a plain list of lists instead of tuples — easier to think about
        entry = [d["index"], d["id"], d["bm25_score"]]
        above.append(entry)

print("documents with score >= " + str(threshold_check) + ": " + f"{len(above):,}")
print("")

#SORTING
#make a copy so we don't mess up the original list
above_to_sort = []
for entry in above:
    above_to_sort.append(entry)

above_sorted = []
while len(above_to_sort) > 0:
    best_pos = 0
    best_score = above_to_sort[0][2]
    for pos in range(len(above_to_sort)):
        if above_to_sort[pos][2] > best_score:
            best_score = above_to_sort[pos][2]
            best_pos = pos
    #move the best entry to the sorted list and remove from unsorted
    above_sorted.append(above_to_sort[best_pos])
    above_to_sort.pop(best_pos)

#PRINT TOP 20 DOCS BY SCORE

print("Top 20 by score:")

top_n = 20
if len(above_sorted) < top_n:
    top_n = len(above_sorted)

for i in range(top_n):
    entry = above_sorted[i]
    doc_index = entry[0]
    doc_id    = entry[1]
    score     = entry[2]
    #rjust pads the index number so columns line up — i looked this up
    index_padded = str(doc_index).rjust(5)
    score_str = f"{score:6.2f}"  #i'm keeping this f-string, formatting floats is annoying otherwise
    print("  " + score_str + "  " + index_padded + "  " + str(doc_id))


#BUILD THE BOUNDARY BAND

print("")

#collect all docs with a score above zero
nonzero_docs = []
for d in proc_docs:
    if d["bm25_score"] > 0:
        entry = [d["index"], d["id"], d["bm25_score"]]
        nonzero_docs.append(entry)

print("total non-zero score docs: " + str(len(nonzero_docs)))

#sort all nonzero docs by score descending 
print("sorting non-zero docs by score (this might take a second)...")

nonzero_to_sort = []
for entry in nonzero_docs:
    nonzero_to_sort.append(entry)

boundary_sorted = []
counter = 0
while len(nonzero_to_sort) > 0:
    best_pos = 0
    best_score = nonzero_to_sort[0][2]
    for pos in range(len(nonzero_to_sort)):
        if nonzero_to_sort[pos][2] > best_score:
            best_score = nonzero_to_sort[pos][2]
            best_pos = pos
    boundary_sorted.append(nonzero_to_sort[best_pos])
    nonzero_to_sort.pop(best_pos)
    counter = counter + 1
    if counter % 1000 == 0:
        print("  sorted " + str(counter) + " docs so far...")

print("sorting done")

#SPLIT INTO ABOVE-BOUNDARY AND BELOW-BOUNDARY LISTS

#docs with score >= boundary (we want the LAST boundary_window of these,
all_above_boundary = []
for entry in boundary_sorted:
    if entry[2] >= boundary:
        all_above_boundary.append(entry)

#docs with score < boundary 
all_below_boundary = []
for entry in boundary_sorted:
    if entry[2] < boundary:
        all_below_boundary.append(entry)

#grab the last boundary_window entries from above (closest to cutoff from above)
above_start = len(all_above_boundary) - boundary_window
if above_start < 0:
    above_start = 0  #in case there are fewer docs above than boundary_window
above_boundary = []
for i in range(above_start, len(all_above_boundary)):
    above_boundary.append(all_above_boundary[i])

#grab the first boundary_window entries from below (closest to cutoff from below)
below_boundary = []
for i in range(len(all_below_boundary)):
    if i >= boundary_window:
        break
    below_boundary.append(all_below_boundary[i])


#COMBINE AND SORT THE BAND (still descending by score)

band_unsorted = []
for entry in above_boundary:
    band_unsorted.append(entry)
for entry in below_boundary:
    band_unsorted.append(entry)

#sort the combined band descending — it's small so this is fast
band_to_sort = []
for entry in band_unsorted:
    band_to_sort.append(entry)

band = []
while len(band_to_sort) > 0:
    best_pos = 0
    best_score = band_to_sort[0][2]
    for pos in range(len(band_to_sort)):
        if band_to_sort[pos][2] > best_score:
            best_score = band_to_sort[pos][2]
            best_pos = pos
    band.append(band_to_sort[best_pos])
    band_to_sort.pop(best_pos)

#PRINT THE BOUNDARY BAND

print("Boundary band (+-" + str(boundary_window) + " around score " + str(boundary) + "):")

for entry in band:
    doc_index = entry[0]
    doc_id    = entry[1]
    score     = entry[2]
    index_padded = str(doc_index).rjust(5)
    score_str = f"{score:6.2f}"
    print("  " + score_str + "  " + index_padded + "  " + str(doc_id))

Summaries

In [ ]:
raw_path = "../articles_raw_proc_data/raw_data/guardian_launder_1999_2026.json"
output_path = "../articles_raw_proc_data/processed_data/summaries_train.json"

#SET UP THE MISTRAL CLIENT

api_key = "REDACTED"
client = Mistral(api_key=api_key)

#LOAD THE RAW CORPUS


print("loading raw corpus from: " + raw_path)
f = open(raw_path, "r", encoding="utf-8")
raw_docs = json.load(f)
f.close()

print("raw corpus loaded: " + f"{len(raw_docs):,}" + " articles")

#build a lookup dictionary: article id -> full article dict
id2doc = {}
for d in raw_docs:
    article_id = d["id"]
    id2doc[article_id] = d

print("id lookup built: " + str(len(id2doc)) + " entries")

#SUMMARIZE FUNCTION

def summarize(doc):
    #pull out the fields we need — .get() returns "" if the field is missing
    title    = doc.get("title", "")
    byline   = doc.get("byline", "")
    bodytext = doc.get("bodyText", "")

    #build the prompt using string concatenation across multiple lines
    prompt = "Title: " + title + "\n"
    prompt = prompt + "Byline: " + byline + "\n\n"
    prompt = prompt + "Body:\n" + bodytext + "\n\n"
    prompt = prompt + "Write a 2-3 sentence summary focused specifically on the anti-money "
    prompt = prompt + "laundering (AML) dimension: what compliance failure occurred, who was "
    prompt = prompt + "involved, and what it reveals about regulatory or institutional weaknesses."

    #call the mistral api — chat.complete sends a single-turn conversation
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )

    #dig into the response object to get the text out
    #.choices is a list, we want the first (and only) choice
    summary_text = response.choices[0].message.content
    return summary_text

#SET UP THE TWO ARTICLE GROUPS WE WANT TO SUMMARIZE

#grab the top 20 from above_sorted 
top20_articles = []
for i in range(20):
    if i < len(above_sorted):  #safety check in case there are fewer than 20
        top20_articles.append(above_sorted[i])

band40_articles = []
for entry in band:
    band40_articles.append(entry)

set_names = ["top20", "band40"]
set_articles = {}
set_articles["top20"]  = top20_articles
set_articles["band40"] = band40_articles

#RUN THE SUMMARIZATION — looping over both groups

#results dict will hold two lists, one per group
results = {}
results["top20"]  = []
results["band40"] = []

for group_index in range(len(set_names)):
    set_name = set_names[group_index]
    articles = set_articles[set_name]

    print("")
    print("── " + set_name + " ──────────────────────────────────────────────────")

    for article_index in range(len(articles)):
        entry = articles[article_index]
        idx   = entry[0]  #corpus position index
        aid   = entry[1]  #article id string
        score = entry[2]  #bm25 score

        score_str = f"{score:.2f}"  
        print("  [" + score_str + "] " + str(aid) + " ... ", end="", flush=True)

        #look up the full article in our id2doc lookup
        doc = id2doc.get(aid)

        #handle the case where the article id isn't in the raw corpus
        #this shouldn't happen but good to be safe
        if doc is None:
            print("NOT FOUND IN RAW CORPUS")

            #build the error entry manually instead of with dict literal unpacking
            error_entry = {}
            error_entry["index"]      = idx
            error_entry["id"]         = aid
            error_entry["bm25_score"] = score
            error_entry["summary"]    = None
            error_entry["error"]      = "id not found"

            results[set_name].append(error_entry)
            #skip to the next article
            continue

        #call the api to get the summary
        summary = summarize(doc)
        print("OK")

        #print first 120 characters of the summary as a preview
        summary_preview = summary[:120]
        print("    " + summary_preview + "...\n")

        #build the result entry for this article
        result_entry = {}
        result_entry["index"]      = idx
        result_entry["id"]         = aid
        result_entry["bm25_score"] = score
        result_entry["title"]      = doc.get("title", "")
        result_entry["byline"]     = doc.get("byline", "")
        result_entry["date"]       = doc.get("date", "")
        result_entry["summary"]    = summary

        results[set_name].append(result_entry)

print("")
print("all summaries done")

#SAVE RESULTS TO DISK

print("saving results to: " + output_path)
f = open(output_path, "w", encoding="utf-8")
json.dump(results, f, ensure_ascii=False, indent=2)
f.close()

#count how many we got in each group
top20_count  = len(results["top20"])
band40_count = len(results["band40"])

print("saved to " + output_path)
print("  top20  : " + str(top20_count) + " articles")
print("  band40 : " + str(band40_count) + " articles")

## EM algorithm experiments

Gaussian gamma fit - random initialization - sweeping over bins 40,60,80,100,120,150

In [ ]:

def em_gamma_normal(x, pi0, k, theta, mu1, sigma1, max_iter=500, tol=1e-8):
    #number of data points
    n = len(x)

    #run up to max_iter iterations of the EM loop
    for iteration in range(max_iter):

        #E STEP: compute responsibilities
        #log probabilities for each component — working in log space to avoid underflow
        #from very low BM25 scores
        log_p0 = np.log(pi0)       + gamma.logpdf(x, a=k, scale=theta)
        log_p1 = np.log(1 - pi0)   + norm.logpdf(x, loc=mu1, scale=sigma1)

        #log of the denominator: log(p0 + p1) computed stably using logaddexp
        #log(p0 + p1) ≠ log(p0) + log(p1)
        #logaddexp(a, b) = log(exp(a) + exp(b)) without overflow 
        #log(exp(a) + exp(b)) = a + log(1 + exp(b - a))
        log_denom = np.logaddexp(log_p0, log_p1)

        #responsibilities: probability each point belongs to component 0 or 1
        #log(p0 / (p0 + p1)) = log(p0) - log(p0 + p1)
        #=log_p0  - log_denom
        r0 = np.exp(log_p0 - log_denom)
        r1 = np.exp(log_p1 - log_denom)

        #M STEP: update parameters using weighted statistics
        #effective number of points assigned to each component
        # doc 1:  r0=0.97,  r1=0.03   ← almost certainly non-relevant
        #doc 2:  r0=0.50,  r1=0.50   ← genuinely uncertain
        #doc 3:  r0=0.08,  r1=0.92   ← almost certainly relevant
        #parameter update in the M step is a weighted average using r0 or r1 as weights.
        n0 = r0.sum()
        n1 = r1.sum()

        #new mixing weight for the gamma component
        pi0_new = n0 / n

        #new normal component mean: weighted average of x using r1 as weights
        weighted_sum_x = 0.0
        for i in range(len(x)):
            weighted_sum_x = weighted_sum_x + r1[i] * x[i]
        mu1_new = weighted_sum_x / n1

        #new normal component standard deviation: weighted variance then sqrt
        weighted_sq_sum = 0.0
        for i in range(len(x)):
            diff = x[i] - mu1_new
            weighted_sq_sum = weighted_sq_sum + r1[i] * diff * diff
        sigma1_new = (weighted_sq_sum / n1) ** 0.5

        #new gamma scale parameter theta
        weighted_sum_x_gamma = 0.0
        for i in range(len(x)):
            weighted_sum_x_gamma = weighted_sum_x_gamma + r0[i] * x[i]
        theta_new = weighted_sum_x_gamma / (n0 * k)

        #new gamma shape parameter k — requires a mini Newton-Raphson loop
        #because there's no closed form solution for k
        #i looked up the update formula: it uses digamma and polygamma functions
        weighted_log_sum = 0.0
        for i in range(len(x)):
            weighted_log_sum = weighted_log_sum + r0[i] * np.log(x[i])
        log_x_bar = weighted_log_sum / n0

        x_bar_gamma = weighted_sum_x_gamma / n0

        k_new = k
        #up to 20 newton steps to find the new k
        for _ in range(20):
            numerator_k   = np.log(k_new) - digamma(k_new) - np.log(x_bar_gamma) + log_x_bar
            denominator_k = (1.0 / k_new) - polygamma(1, k_new)
            k_new = k_new - numerator_k / denominator_k
            #k must be positive — if it goes negative something went wrong
            if k_new <= 0:
                k_new = 1e-3
                break

        #CHECK CONVERGENCE -----
        #measure the largest change in any parameter this iteration
        delta_pi0    = abs(pi0_new    - pi0)
        delta_mu1    = abs(mu1_new    - mu1)
        delta_sigma1 = abs(sigma1_new - sigma1)
        delta_theta  = abs(theta_new  - theta)
        delta_k      = abs(k_new      - k)

        #find the max delta manually
        delta = delta_pi0
        if delta_mu1    > delta: delta = delta_mu1
        if delta_sigma1 > delta: delta = delta_sigma1
        if delta_theta  > delta: delta = delta_theta
        if delta_k      > delta: delta = delta_k

        #update all parameters for the next iteration
        pi0    = pi0_new
        mu1    = mu1_new
        sigma1 = sigma1_new
        theta  = theta_new
        k      = k_new

        #stop early if converged
        if delta < tol:
            break

    return pi0, k, theta, mu1, sigma1, r1



#WRAPPER FUNCTION: run EM many times with random starts, keep the best fit
#random restarts help avoid getting stuck in a bad local optimum


def run_em_gamma_normal_ll(x, n_runs=100, max_iter=500, tol=1e-8, n_bins=80):

    #compute the histogram we'll use to measure goodness of fit
    hist_result  = np.histogram(x, bins=n_bins, density=True)
    counts       = hist_result[0]
    bin_edges    = hist_result[1]

    #bin centers = midpoint of each bin edge pair
    bin_centers = []
    for i in range(len(bin_edges) - 1):
        center = 0.5 * (bin_edges[i] + bin_edges[i + 1])
        bin_centers.append(center)
    bin_centers = np.array(bin_centers)

    #tracking the best fit across all random restarts
    best_gof    = float("inf")  #lower is better-starts at infinity
    best_params = None
    best_r1     = None
    n_rejected  = 0  #count of runs we threw out because mu1 <= gamma mean

    np.random.seed(123) 

    for run in range(n_runs):
        #random starting values for all parameters but guided by IR considerations
        pi0    = np.random.uniform(0.60, 0.95)  #most docs are probably non-relevant
        k      = np.random.uniform(0.5,  3.0)
        theta  = np.random.uniform(1.0,  5.0)
        #mu1 starts somewhere in the upper quartile of scores
        mu1    = np.random.uniform(np.percentile(x, 75), x.max())
        sigma1 = np.random.uniform(1.0, 10.0)

        #try running EM — catch any numerical errors and skip this run
        try:
            fit_result = em_gamma_normal(x, pi0, k, theta, mu1, sigma1, max_iter, tol)
        except Exception:
            continue

        #unpack the fit result
        pi0_fit    = fit_result[0]
        k_fit      = fit_result[1]
        theta_fit  = fit_result[2]
        mu1_fit    = fit_result[3]
        sigma1_fit = fit_result[4]
        r1         = fit_result[5]

        #sanity check: the normal (relevant) component mean must be ABOVE the gamma mean
        #if it's not, the two components have flipped and the fit is meaningless
        gamma_mean = k_fit * theta_fit
        if mu1_fit <= gamma_mean:
            n_rejected = n_rejected + 1
            continue

        #compute goodness of fit: sum of squared differences between
        #the fitted mixture density and the actual histogram counts
        mixture_vals_run = []
        for i in range(len(bin_centers)):
            bc = bin_centers[i]
            gamma_part  = pi0_fit        * gamma.pdf(bc, a=k_fit,  scale=theta_fit)
            normal_part = (1 - pi0_fit)  * norm.pdf(bc,  loc=mu1_fit, scale=sigma1_fit)
            mixture_vals_run.append(gamma_part + normal_part)
        mixture_vals_run = np.array(mixture_vals_run)

        #sum of squared residuals
        sq_diff_sum = 0.0
        for i in range(len(mixture_vals_run)):
            diff = mixture_vals_run[i] - counts[i]
            sq_diff_sum = sq_diff_sum + diff * diff
        gof = sq_diff_sum

        #keep this run if it's the best so far
        if gof < best_gof:
            best_gof = gof

            #store all best params in a plain dict — building it field by field
            best_params = {}
            best_params["pi0"]        = pi0_fit
            best_params["pi1"]        = 1 - pi0_fit
            best_params["k"]          = k_fit
            best_params["theta"]      = theta_fit
            best_params["mu1"]        = mu1_fit
            best_params["sigma1"]     = sigma1_fit
            best_params["gamma_mean"] = gamma_mean
            best_params["gof"]        = gof
            best_params["run"]        = run + 1  #+1 so run numbers are 1-indexed in output

            #.copy() so we don't accidentally overwrite r1 in a later iteration
            best_r1 = r1.copy()

    #if every single run was rejected or crashed, return None values
    if best_params is None:
        return None, None, bin_centers, counts, None, None, None

    #store how many runs got rejected in the params dict for reporting
    best_params["n_rejected"] = n_rejected

    #compute the three density curves for the best fit (for plotting)
    p = best_params

    mixture_vals_list     = []
    nonrelevant_vals_list = []
    relevant_vals_list    = []

    for i in range(len(bin_centers)):
        bc = bin_centers[i]

        gamma_part      = p["pi0"] * gamma.pdf(bc, a=p["k"],   scale=p["theta"])
        normal_part     = p["pi1"] * norm.pdf(bc,  loc=p["mu1"], scale=p["sigma1"])

        mixture_vals_list.append(gamma_part + normal_part)
        nonrelevant_vals_list.append(gamma_part)
        relevant_vals_list.append(normal_part)

    mixture_vals     = np.array(mixture_vals_list)
    nonrelevant_vals = np.array(nonrelevant_vals_list)
    relevant_vals    = np.array(relevant_vals_list)

    return best_params, best_r1, bin_centers, counts, mixture_vals, nonrelevant_vals, relevant_vals




Getting parameter estimations for different bin widths and plot for bin width = 80

In [ ]:


#get the non-zero scores for the enriched query
scores_all = all_scores["Enriched query with low-recall, high-precision set"]

#collect only scores above zero — doing this manually
nz_list = []
for s in scores_all:
    if s > 0:
        nz_list.append(s)
nz = np.array(nz_list)

print("non-zero scores to fit: " + f"{len(nz):,}")

#the bin counts we want to test
bins_sweep = [40, 60, 80, 100, 120, 150]

n_runs     = 100
x_max_plot = 60  #we'll only plot up to this score value (the tail is uninteresting)

#the one bin count we actually want to plot
plot_bins = 80

#---------------------------------------------------------------------------
#FIT FOR ALL BIN COUNTS AND PRINT PARAMETERS
#we save the fit for plot_bins separately so we can plot it afterwards
#---------------------------------------------------------------------------

#we'll store the plot_bins fit result here once we hit it in the loop
saved_params           = None
saved_r1               = None
saved_bin_centers      = None
saved_counts           = None
saved_mixture_vals     = None
saved_nonrelevant_vals = None
saved_relevant_vals    = None

print("")
print("=== parameter sweep across bin counts ===")

for sweep_index in range(len(bins_sweep)):
    n_bins = bins_sweep[sweep_index]

    print("fitting n_bins=" + str(n_bins) + " ...")

    result = run_em_gamma_normal_ll(nz, n_runs=n_runs, n_bins=n_bins)

    #unpack result tuple by position
    params           = result[0]
    r1               = result[1]
    bin_centers      = result[2]
    counts           = result[3]
    mixture_vals     = result[4]
    nonrelevant_vals = result[5]
    relevant_vals    = result[6]

    #if no valid fit was found, skip this bin count
    if params is None:
        print("  n_bins=" + str(n_bins) + ": no valid fit found")
        continue

    #print a summary line for this fit — we do this for ALL bin counts
    print("  n_bins=" + str(n_bins) +
          "  pi0="    + str(round(params["pi0"],    4)) +
          "  k="      + str(round(params["k"],      4)) +
          "  theta="  + str(round(params["theta"],  4)) +
          "  mu1="    + str(round(params["mu1"],    4)) +
          "  sigma1=" + str(round(params["sigma1"], 4)) +
          "  gof="    + str(round(params["gof"],    8)) +
          "  rejected=" + str(params["n_rejected"]))

    #if this is the bin count we want to plot, save the results for later
    if n_bins == plot_bins:
        saved_params           = params
        saved_r1               = r1
        saved_bin_centers      = bin_centers
        saved_counts           = counts
        saved_mixture_vals     = mixture_vals
        saved_nonrelevant_vals = nonrelevant_vals
        saved_relevant_vals    = relevant_vals
        print("  ^ saved this fit for plotting")

print("")
print("parameter sweep done")

#---------------------------------------------------------------------------
#CHECK WE ACTUALLY GOT A FIT FOR THE PLOT BIN COUNT
#---------------------------------------------------------------------------

if saved_params is None:
    print("ERROR: no valid fit found for n_bins=" + str(plot_bins) + ", cannot plot")
else:
    print("plotting fit for n_bins=" + str(plot_bins) + " ...")

    #bin width = distance between adjacent bin centers
    bin_width = float(saved_bin_centers[1] - saved_bin_centers[0])

    #build annotation text for the plot
    annotation_text = (
        "π₀=" + str(round(saved_params["pi0"], 3)) + "<br>" +
        "k="  + str(round(saved_params["k"],   3)) +
        ", θ=" + str(round(saved_params["theta"], 3)) + "<br>" +
        "μ₁=" + str(round(saved_params["mu1"],   3)) +
        ", σ₁=" + str(round(saved_params["sigma1"], 3)) + "<br>" +
        "GOF=" + str(round(saved_params["gof"], 6))
    )

    #single subplot — just one plot now instead of a grid
    fig = make_subplots(rows=1, cols=1,
                        subplot_titles=["n_bins=" + str(plot_bins)])

    #histogram of the actual data
    fig.add_trace(
        go.Bar(
            x=saved_bin_centers,
            y=saved_counts,
            width=bin_width,
            marker_color="#aec7e8",
            opacity=0.6,
            name="data"
        ),
        row=1, col=1
    )

    #fitted mixture curve (total)
    fig.add_trace(
        go.Scatter(
            x=saved_bin_centers,
            y=saved_mixture_vals,
            mode="lines",
            line=dict(color="black", width=2),
            name="mixture"
        ),
        row=1, col=1
    )

    #gamma component (non-relevant)
    fig.add_trace(
        go.Scatter(
            x=saved_bin_centers,
            y=saved_nonrelevant_vals,
            mode="lines",
            line=dict(color="#d62728", width=1, dash="dash"),
            name="gamma (non-relevant)"
        ),
        row=1, col=1
    )

    #normal component (relevant)
    fig.add_trace(
        go.Scatter(
            x=saved_bin_centers,
            y=saved_relevant_vals,
            mode="lines",
            line=dict(color="#2ca02c", width=1, dash="dash"),
            name="normal (relevant)"
        ),
        row=1, col=1
    )

    #add annotation in top right corner
    fig.add_annotation(
        text=annotation_text,
        xref="x", yref="y",
        x=x_max_plot * 0.98,
        y=max(saved_counts) * 0.98,
        showarrow=False,
        font=dict(size=10),
        xanchor="right",
        yanchor="top"
    )

    fig.update_xaxes(range=[0, x_max_plot], row=1, col=1)

    fig.update_layout(
        height=500,
        width=800,
        title_text="EM gamma-normal mixture fit (n_bins=" + str(plot_bins) + ")"
    )

    fig.write_image("em_mixture_fit.png", width=800, height=500, scale=2)
    fig.show()

    print("plot saved to em_mixture_fit.png")

Likelihood ratio test with one-component Gamma distribution

In [ ]:

#LOG-LIKELIHOOD FUNCTION FOR THE TWO-COMPONENT MIXTURE

def log_likelihood_gn(x, pi0, k, theta, mu1, sigma1):
    log_p0 = np.log(pi0)     + gamma.logpdf(x, a=k, scale=theta)
    log_p1 = np.log(1 - pi0) + norm.logpdf(x, loc=mu1, scale=sigma1)

    log_likelihoods_per_point = np.logaddexp(log_p0, log_p1)

    total_ll = 0.0
    for ll_val in log_likelihoods_per_point:
        total_ll = total_ll + ll_val

    return total_ll

#TWO-COMPONENT FIT: random restarts, keep best by log-likelihood

def run_em_gamma_normal_ll(x, n_runs=100, max_iter=500, tol=1e-8):
    #no histogram / bin count needed here — we select by ll not gof
    best_ll     = -float("inf")
    best_params = None
    best_r1     = None
    n_rejected  = 0

    np.random.seed(123)

    for run in range(n_runs):
        pi0    = np.random.uniform(0.60, 0.95)
        k      = np.random.uniform(0.5,  3.0)
        theta  = np.random.uniform(1.0,  5.0)
        mu1    = np.random.uniform(np.percentile(x, 75), x.max())
        sigma1 = np.random.uniform(1.0, 10.0)

        try:
            fit_result = em_gamma_normal(x, pi0, k, theta, mu1, sigma1, max_iter, tol)
        except Exception:
            continue

        pi0_fit    = fit_result[0]
        k_fit      = fit_result[1]
        theta_fit  = fit_result[2]
        mu1_fit    = fit_result[3]
        sigma1_fit = fit_result[4]
        r1         = fit_result[5]

        #reject runs where the normal component mean is below the gamma mean
        #that means the two components have swapped and the fit is meaningless
        gamma_mean = k_fit * theta_fit
        if mu1_fit <= gamma_mean:
            n_rejected = n_rejected + 1
            continue

        ll = log_likelihood_gn(x, pi0_fit, k_fit, theta_fit, mu1_fit, sigma1_fit)

        if ll > best_ll:
            best_ll = ll

            best_params = {}
            best_params["pi0"]        = pi0_fit
            best_params["pi1"]        = 1 - pi0_fit
            best_params["k"]          = k_fit
            best_params["theta"]      = theta_fit
            best_params["mu1"]        = mu1_fit
            best_params["sigma1"]     = sigma1_fit
            best_params["gamma_mean"] = gamma_mean
            best_params["ll"]         = ll
            best_params["run"]        = run + 1

            best_r1 = r1.copy()

        if run % 20 == 0 and run > 0:
            print("  completed " + str(run) + " restarts so far...")

    if best_params is None:
        return None, None

    best_params["n_rejected"] = n_rejected
    return best_params, best_r1


#SINGLE GAMMA FIT (null model for the lrt)


def fit_single_gamma(x):
    x_mean = 0.0
    for val in x:
        x_mean = x_mean + val
    x_mean = x_mean / len(x)

    sq_diff_sum = 0.0
    for val in x:
        diff = val - x_mean
        sq_diff_sum = sq_diff_sum + diff * diff
    x_var = sq_diff_sum / len(x)

    k     = (x_mean * x_mean) / x_var
    theta = x_var / x_mean

    log_x_sum = 0.0
    for val in x:
        log_x_sum = log_x_sum + np.log(val)
    log_x_bar = log_x_sum / len(x)

    for _ in range(100):
        numerator_k   = np.log(k) - digamma(k) - np.log(x_mean) + log_x_bar
        denominator_k = (1.0 / k) - polygamma(1, k)
        k_new = k - numerator_k / denominator_k
        if k_new <= 0:
            k_new = 1e-3
            break
        if abs(k_new - k) < 1e-10:
            break
        k = k_new

    theta = x_mean / k

    log_pdf_vals = gamma.logpdf(x, a=k, scale=theta)
    ll = 0.0
    for val in log_pdf_vals:
        ll = ll + val

    return k, theta, ll


#RUN EVERYTHING

#get non-zero scores
scores_all = all_scores["Enriched query with low-recall, high-precision set"]
nz_list = []
for s in scores_all:
    if s > 0:
        nz_list.append(s)
nz = np.array(nz_list)

print("non-zero scores: " + f"{len(nz):,}")
print("")

#fit two-component model
print("fitting two-component gamma-normal mixture (" + str(100) + " random restarts)...")
two_result = run_em_gamma_normal_ll(nz, n_runs=100)

params = two_result[0]
r1     = two_result[1]

if params is None:
    print("no valid fit found — try increasing n_runs")

else:
    print("")
    print("Two-component Gamma-Gaussian:")
    print("  pi0="    + str(round(params["pi0"],        4)))
    print("  k="      + str(round(params["k"],          4)) +
          "  theta="  + str(round(params["theta"],      4)) +
          "  gamma_mean=" + str(round(params["gamma_mean"], 4)))
    print("  mu1="    + str(round(params["mu1"],        4)) +
          "  sigma1=" + str(round(params["sigma1"],     4)))
    print("  ll="     + str(round(params["ll"],         4)) +
          "  best run=" + str(params["run"]))
    print("  rejected=" + str(params["n_rejected"]) +
          "  total=100")

    #fit single gamma
    print("")
    print("fitting single gamma (null model)...")
    single_result = fit_single_gamma(nz)
    k_single     = single_result[0]
    theta_single = single_result[1]
    ll_single    = single_result[2]

    print("Single Gamma:")
    print("  k="     + str(round(k_single,     4)) +
          "  theta=" + str(round(theta_single, 4)) +
          "  ll="    + str(round(ll_single,    4)))

    #likelihood ratio test
    #H0: single gamma is good enough
    #H1: two-component mixture fits significantly better
    #df = 5 params (two-component) - 2 params (single gamma) = 3
    ll_two      = params["ll"]
    lambda_stat = -2.0 * (ll_single - ll_two)
    df          = 3
    p_val       = chi2.sf(lambda_stat, df)

    print("")
    print("Likelihood ratio test:")
    print("  Lambda=" + str(round(lambda_stat, 4)) +
          "  df="     + str(df) +
          "  p-value=" + str(round(p_val, 6)))

    if p_val < 0.05:
        print("  -> two-component model significantly better than single gamma (p<0.05)")
    else:
        print("  -> no significant improvement over single gamma (p>=0.05)")

## Optimizing threshold through F1 score

In [ ]:

#this cell sweeps over every possible score threshold and computes the
#expected precision, recall, and f1 we would get if we used that threshold
#to decide which documents are relevant


#pull the fitted parameters out of the params dict one by one
#easier to read than params["pi0"] everywhere
pi0    = params["pi0"]
k      = params["k"]
theta  = params["theta"]
mu1    = params["mu1"]
sigma1 = params["sigma1"]

#omega = proportion of documents we expect to be relevant
#1 minus the gamma component weight
omega = 1 - pi0

#total number of scored documents and expected number of relevant ones
total_docs    = len(nz)
expected_relevant = omega * total_docs  

print("total scored documents: " + str(total_docs))
print("expected relevant documents (R): " + str(round(expected_relevant, 2)))
print("omega (proportion relevant): "     + str(round(omega, 4)))


#SWEEP OVER EVERY UNIQUE SCORE AS A CANDIDATE THRESHOLD
#the treshold candidates are all retrieved scores
#for each threshold t: if we retrieve all docs with score >= t,
#how good would that be?


#we collect results in plain lists and convert to arrays at the end
precisions_list = []
recalls_list    = []
f1s_list        = []
thresholds_list = []

#sort the scores so we evaluate thresholds in ascending order
sorted_scores = np.sort(nz)

print("sweeping over " + str(len(sorted_scores)) + " threshold candidates...")

for i in range(len(sorted_scores)):
    t = sorted_scores[i]

    #progress print every 5000 thresholds so we know it's still running
    if i % 5000 == 0 and i > 0:
        print("  evaluated " + str(i) + " thresholds so far...")

    #expected true positives: relevant docs that score above t
    #norm.cdf(t) = probability a relevant doc scores BELOW t
    #so 1 - norm.cdf(t) = probability it scores ABOVE t 
    prob_relevant_above_t   = 1 - norm.cdf(t,  loc=mu1,  scale=sigma1)
    expected_tp             = expected_relevant * prob_relevant_above_t

    #expected false positives: non-relevant docs that score above t
    #same logic but using the gamma distribution for the non-relevant component
    num_nonrelevant         = total_docs - expected_relevant
    prob_nonrelevant_above_t = 1 - gamma.cdf(t, a=k, scale=theta)
    expected_fp             = num_nonrelevant * prob_nonrelevant_above_t

    #total expected retrieved documents at this threshold
    expected_retrieved = expected_tp + expected_fp

    #skip this threshold if nothing would be retrieved — can't divide by zero
    if expected_retrieved == 0:
        continue

    #precision = what fraction of retrieved docs are actually relevant
    prec = expected_tp / expected_retrieved

    #recall = what fraction of all relevant docs we managed to retrieve
    rec = expected_tp / expected_relevant

    #f1 = harmonic mean of precision and recall
    #check prec + rec > 0 to avoid dividing by zero (both being zero is theoretically
    #possible at extreme thresholds)
    if (prec + rec) > 0:
        f1 = 2 * prec * rec / (prec + rec)
    else:
        f1 = 0.0

    precisions_list.append(prec)
    recalls_list.append(rec)
    f1s_list.append(f1)
    thresholds_list.append(t)

print("sweep done, evaluated " + str(len(thresholds_list)) + " valid thresholds")

#convert lists to numpy arrays so we can do array operations below
precisions = np.array(precisions_list)
recalls    = np.array(recalls_list)
f1s        = np.array(f1s_list)
thresholds = np.array(thresholds_list)

#FIND THE THRESHOLD THAT MAXIMISES F1

best_idx   = 0
best_f1_so_far = f1s[0]

for i in range(len(f1s)):
    if f1s[i] > best_f1_so_far:
        best_f1_so_far = f1s[i]
        best_idx       = i

#pull out the best values using the index we just found
best_thr  = thresholds[best_idx]
best_f1   = f1s[best_idx]
best_prec = precisions[best_idx]
best_rec  = recalls[best_idx]

#count how many documents actually score at or above the best threshold
docs_above_threshold = 0
for s in nz:
    if s >= best_thr:
        docs_above_threshold = docs_above_threshold + 1


print("")
print("F1-optimal threshold  : " + str(round(best_thr,  4)))
print("  Expected precision  : " + str(round(best_prec, 4)))
print("  Expected recall     : " + str(round(best_rec,  4)))
print("  Expected F1         : " + str(round(best_f1,   4)))
print("  Documents above     : " + str(docs_above_threshold))